Step 1: Import Libraries


In [3]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

from xgboost import XGBClassifier

Step 2: Load and Explore Data


In [4]:
# Load Datasets
ipo = pd.read_csv("ipo.csv")
cleaned_ipo = pd.read_csv("cleaned_ipo_data 2022-25.csv")

# Display First Few Rows
print(ipo.head())
print(cleaned_ipo.head(380))

# Check for Missing Values
print(ipo.isnull().sum())
print(cleaned_ipo.isnull().sum())

   Unnamed: 0 Unnamed: 0_level_0    Issue Details          Issue Details.1  \
0         NaN               Date         IPO Name  Issue Size  (in crores)   
1         0.0         17-10-2022  Electronics Mar                      500   
2         1.0         26-09-2022  Harsha Engineer                      755   
3         2.0         15-09-2022              TMB                      792   
4         3.0         06-09-2022  Dreamfolks Serv                    562.1   

  Subscription Subscription.1 Subscription.2 Subscription.3  Price  \
0          QIB            HNI            RII          Total  Issue   
1        58.81          15.39           8.27          24.23     59   
2       113.82          40.36          12.44          47.19    330   
3         0.51           1.77           3.44           1.39    525   
4        27.48          14.18          24.19          23.25    326   

        Price.1        Price.2           Price.3 Price.4             Price.5  
0  Listing Open  Listing Close 

Combine datasets


In [ ]:
from fuzzywuzzy import process

def fuzzy_merge(df1, df2, key1, key2, threshold=80):
    """
    Improved fuzzy matching function with error handling and debugging
    """
    
    # Convert to string type to avoid comparison issues
    df1[key1] = df1[key1].astype(str)
    df2[key2] = df2[key2].astype(str)
    
    matches = []
    scores = []
    
    print(f"Starting fuzzy merge between {key1} and {key2}...")
    
    for i, name in enumerate(df1[key1]):
        try:
            if i % 50 == 0:  # Print progress every 50 rows
                print(f"Processing row {i}/{len(df1)}")
                
            # Skip empty names
            if not name or name.lower() in ['nan', 'none', '']:
                matches.append(None)
                scores.append(0)
                continue
                
            match, score = process.extractOne(name, df2[key2].unique())
            
            if score >= threshold:
                matches.append(match)
                scores.append(score)
            else:
                matches.append(None)
                scores.append(score)
                
        except Exception as e:
            print(f"Error processing row {i} ('{name}'): {str(e)}")
            matches.append(None)
            scores.append(0)
    
    df1['matched_name'] = matches
    df1['match_score'] = scores
    
    print("\nMatch quality summary:")
    print(f"Successful matches: {len([x for x in matches if x is not None])}")
    print(f"Average score: {np.mean(scores):.1f}")
    print(f"Median score: {np.median(scores):.1f}")
    
    return df1.merge(df2, left_on='matched_name', right_on=key2, how='inner')

# Run with error handling
try:
    print("Attempting fuzzy merge...")
    combined = fuzzy_merge(ipo, cleaned_ipo, 'IPO Name', 'Name', threshold=80)
    
    print("\nMerge successful! First few matches:")
    print(combined[['IPO Name', 'Name', 'match_score']].head())
    
    print("\nCombined dataset shape:", combined.shape)
    print("\nSample of combined data:")
    print(combined.head())
    
except Exception as e:
    print(f"\nFuzzy merge failed: {str(e)}")
    print("\nAttempting simple exact match as fallback...")
    combined = pd.merge(ipo, cleaned_ipo, left_on='IPO Name', right_on='Name', how='inner')
    print("\nSimple merge result shape:", combined.shape)

Attempting fuzzy merge...

Fuzzy merge failed: 'IPO Name'

Attempting simple exact match as fallback...


KeyError: 'IPO Name'

 preprocess the data.

In [7]:
# Target Variable (Binary)
combined['Target'] = ((combined['Listing Gains(%)'].fillna(0) > 0) | (combined['Returns'].fillna(0) > 0)).astype(int)

# Log Transform Skewed Features
combined['Log_Issue_Size'] = np.log1p(combined['Issue Size (in crores)'])

# Investor Category (Categorical)
combined['Investor_Category'] = pd.cut(
    combined['RII'], 
    bins=[-np.inf, 50, 100, np.inf], 
    labels=['Retail', 'HNI', 'Institutional']
)

# Drop Unnecessary Columns
combined = combined[['Log_Issue_Size', 'Total', 'Investor_Category', 'Target']]

NameError: name 'combined' is not defined

Step 5: Train-Test Split

In [8]:
# Define Features and Target
X = combined.drop(columns=['Target'])
y = combined['Target']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

NameError: name 'combined' is not defined

Step 6: Build Pipeline

In [9]:
# Preprocessing Pipelines
numerical_features = ['Log_Issue_Size', 'Total']
categorical_features = ['Investor_Category']

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

# Full Pipeline with Model
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42))
])

Step 7: Hyperparameter Tuning

In [10]:
# Hyperparameter Grid
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2]
}

# Grid Search
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='roc_auc')
grid_search.fit(X_train, y_train)

# Best Parameters
print("Best Parameters:", grid_search.best_params_)

NameError: name 'X_train' is not defined

Step 8: Evaluate the Model

In [11]:
# Predict Probabilities
y_pred_proba = grid_search.predict_proba(X_test)[:, 1]

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC: {roc_auc:.2f}")

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.legend()
plt.show()

# Classification Report
y_pred = grid_search.predict(X_test)
print(classification_report(y_test, y_pred))

NameError: name 'X_test' is not defined

Step 9: Deploy the Model

In [12]:
# Save Model
import joblib
joblib.dump(grid_search, "ipo_model.pkl")



['ipo_model.pkl']

Step 10: Run the Streamlit App

In [ ]:
streamlit run app.py

SyntaxError: invalid syntax (3737097518.py, line 1)